# DTSC-580: Data Manipulation
## Assignment: Survivor

### Name:

## Overview

In this assignment, you will test all the skills that you have learned during this course to manipulate the provided data to find the answers to questions about the TV show Survivor.  If you are not familiar with this show, start by watching this short clip that briefly explains it:  [Survivor Explained](https://www.youtube.com/watch?v=l1-hTpG_krk)

Please note that your notebook should be named `survivor` when submitting to CodeGrade for the automatic grading to work properly.

## Data

The Survivor data is a R package from Daniel Oehm.  Daniel has made the data for this package available as an Excel file as explained in his article on [gradientdescending.com](http://gradientdescending.com/survivor-data-from-the-tv-series-in-r/).  Please make sure that you use the file from our Brightspace page though to make sure that your data will match what CodeGrade is expecting.  We have also updated some errors in the file, which is another reason that you must use the data given to you.

You need to first read the article on the website linked above.  This will give you additional details about the data that will be important as you answer the questions below.

Please note that there is a data dictionary in the file that explains the columns in the data.  You will also want to become familiar with the various spreadsheets and column names.  This will help you out tremendously in this assignment.

Finally, here are a couple of things to know for those of you that have not seen the show:
- Survivor is a reality TV show that first aired May 31, 2000 and is currently still on TV.
- Contestants are broken up into two teams (usually) where they live in separate camps. 
- The teams compete in various challenges for rewards (food, supplies, brief experience trips, etc) and tribal immunity.  
- The team that loses a challenge, and therefore doesn't get the tribal immunity, goes to tribal council where they have to vote one of their members out (this data is represented in the "Vote History" spreadsheet).
- After there are a small number of contestants left, the tribes are merged into one tribe where each contestant competes for individual immunity.  The winner of the individual immunity cannot get voted out and is safe at the next tribal council.
- The are also hidden immunity idols that are hidden around the campground.  If a contestant finds and plays their hidden immunity at the tribal council, then all votes against them do not count, and the player with the next highest number of votes goes home.
- When the contestants get down to 2 or 3 people, a number of the last contestants, known as the jury, come back to vote for the person who they think should win the game.  The winner is the one who gets the most jury votes (this data is represented in the "Jury Votes" spreadsheet).  This person is known as the Sole Survivor.
- Voting recap: 
    - Tribal Council votes (Vote History spreadsheet) are bad; contestants with the most votes get sent home
    - Jury Votes (Jury Votes spreadsheet) are good; contestants with the most votes win the game and is the Sole Survivor

## Note

<u>Show Work</u>

Remember that you must show your work.  Students submissions are spot checked manually to verify that they are not hard coding the answer from looking only in the file or in CodeGrade's expected output.  If this is seen, the student's answer will be manually marked wrong and their grade will be changed to reflect this.

For example, if the question is who is the contestant who has received the most tribal votes to be voted out.  Select their record from the `castaway_details` DataFrame.  

You would show your work and code similar to this:
```
### incorrect way ###
Q1 = castaway_details[castaway_details['castaway_id'] ==  333]

### correct way - showing your work ###
# get index
idx = vote_history.groupby('vote_id').size().sort_values(ascending=False).index[0]

# select row based on index 
Q1 = castaway_details[castaway_details['castaway_id'] ==  idx]
```

<u>Use Copy</u>

Don't change any of the original DataFrames unless specifically asked or CodeGrade will not work correctly for this assignment.  Make sure you use `copy()` if needed.

In [1]:
# standard imports
import pandas as pd
import numpy as np

# Do not change this option; This allows the CodeGrade auto grading to function correctly
pd.set_option('display.max_columns', None)

First, import the data from the `survivor.xlsx` file, calling the respective DataFrames the same as the sheet name but with lowercase and [snake case](https://en.wikipedia.org/wiki/Snake_case).  For example, the sheet called `Castaway Details` should be saved as a DataFrame called `castaway_details`.  Make sure that the data files are in the same folder as your notebook.

Note:  You may or may not need to install [openpyxl](https://openpyxl.readthedocs.io/en/stable/) for the code below to work.  You can use: `$ pip install openpyxl`

In [2]:
# import data from Excel

# setup Filename and Object
fileName = "survivor.xlsx"
xls = pd.ExcelFile(fileName)

# import individual sheets
castaway_details = pd.read_excel(xls, 'Castaway Details')
castaways = pd.read_excel(xls, 'Castaways')
challenge_description = pd.read_excel(xls, 'Challenge Description')
challenge_results = pd.read_excel(xls, 'Challenge Results')
confessionals = pd.read_excel(xls, 'Confessionals')
hidden_idols = pd.read_excel(xls, 'Hidden Idols')
jury_votes = pd.read_excel(xls, 'Jury Votes')
tribe_mapping = pd.read_excel(xls, 'Tribe Mapping')
viewers = pd.read_excel(xls, 'Viewers')
vote_history = pd.read_excel(xls, 'Vote History')
season_summary = pd.read_excel(xls, 'Season Summary')
season_palettes = pd.read_excel(xls, 'Season Palettes')
tribe_colours = pd.read_excel(xls, 'Tribe Colours')

In [3]:
season_summary

,Season Name,Season,Location,Country,Tribe Setup,Full Name,Winner Id,Winner,Runner Ups,Final Vote,Timeslot,Premiered,Ended,Filming Started,Filming Ended,Viewers Premier,Viewers Finale,Viewers Reunion,Viewers Mean,Rank
0,Survivor: Borneo,1,"Pulau Tiga, Sabah, Malaysia",Malaysia,Two tribes of eight new players,Richard Hatch,16,Richard,Kelly Wiglesworth,4-3,Wednesday 8:00 pm,2000-05-31,2000-08-23,2000-03-13,2000-04-20,15.51,51.69,36.70,28.30,2.0
1,Survivor: The Australian Outback,2,"Herbert River at Goshen Station, Queensland, A...",Australia,Two tribes of eight new players,Tina Wesson,32,Tina,Colby Donaldson,4-3,Thursday 8:00 pm,2001-01-28,2001-05-03,2000-10-23,2000-12-03,45.37,36.35,28.01,29.80,1.0
2,Survivor: Africa,3,"Shaba National Reserve, Kenya",Kenya,Two tribes of eight new players,Ethan Zohn,48,Ethan,Kim Johnson,5-2,Thursday 8:00 pm,2001-10-11,2002-01-10,2001-07-11,2001-08-18,23.84,27.26,19.05,20.69,8.0
3,Survivor: Marquesas,4,"Nuku Hiva, Marquesas Islands, French Polynesia",Polynesia,Two tribes of eight new players,Vecepia Towery,64,Vecepia,Neleh Dennis,4-3,Thursday 8:00 pm,2002-02-28,2002-05-19,2001-11-12,2001-12-20,23.19,25.87,19.05,20.77,6.0
4,Survivor: Thailand,5,"Ko Tarutao, Satun Province, Thailand",Thailand,Two tribes of eight new players; picked by the...,Brian Heidik,80,Brian,Clay Jordan,4-3,Thursday 8:00 pm,2002-09-19,2002-12-19,2002-06-10,2002-07-18,23.05,24.08,20.43,21.21,4.0
5,Survivor: The Amazon,6,"Rio Negro, Amazonas, Brazil",Brazil,Two tribes of eight new players divided by gender,Jenna Morasca,96,Jenna,Matthew Von Ertfelda,6-1,Thursday 8:00 pm,2003-02-13,2003-05-11,2002-11-07,2002-12-15,23.26,22.29,20.43,19.97,9.0
6,Survivor: Pearl Islands,7,"Pearl Islands, Panama",Panama,Two tribes of eight new players,Sandra Diaz-Twine,112,Sandra,Lillian Morris,6-1,Thursday 8:00 pm,2003-09-18,2003-12-14,2003-06-23,2003-07-31,21.50,25.23,21.87,20.72,7.0
7,Survivor: All-Stars,8,"Pearl Islands, Panama",Panama,Three tribes of six returning players,Amber Brkich,27,Amber,Rob Mariano,4-3,Thursday 8:00 pm,2004-02-01,2004-05-09,2003-11-03,2003-12-11,33.53,24.76,21.87,21.49,3.0
8,Survivor: Vanuatu,9,"Efate, Shefa, Vanuatu",Vanuatu,Two tribes of nine new players divided by gender,Chris Daugherty,130,Chris,Twila Tanner,5-2,Thursday 8:00 pm,2004-09-16,2004-12-12,2004-06-28,2004-08-05,20.06,19.72,15.23,19.64,10.0
9,Survivor: Palau,10,"Koror, Palau",Palau,A schoolyard pick of two tribes of nine new pl...,Tom Westman,150,Tom,Katie Gallagher,6-1,Thursday 8:00 pm,2005-02-17,2005-05-15,2004-10-27,2004-12-04,23.66,20.80,15.23,20.91,5.0


In [4]:
castaways.head()

,Season Name,Season,Full Name,Castaway Id,Castaway,Age,City,State,Personality Type,Episode,Day,Order,Result,Jury Status,Original Tribe,Swapped Tribe,Swapped Tribe 2,Merged Tribe,Total Votes Received,Immunity Idols Won
0,Survivor: 41,41,Erika Casupanan,594,Erika,32,Toronta,Ontario,ENFP,13,26,18,Sole Survivor,NaN,Luvu,NaN,NaN,Via Kana,2,8
1,Survivor: 41,41,Deshawn Radden,601,Deshawn,26,Miami,Florida,ENTP,13,26,17,Runner-up,NaN,Luvu,NaN,NaN,Via Kana,7,6
2,Survivor: 41,41,Xander Hastings,597,Xander,20,Chicago,Illinois,INFJ,13,26,16,2nd runner-up,NaN,Yase,NaN,NaN,Via Kana,2,6
3,Survivor: 41,41,Heather Aldret,593,Heather,52,Charleston,South Carolina,ISFJ,13,25,15,15th voted out,8th jury member,Luvu,NaN,NaN,Via Kana,4,6
4,Survivor: 41,41,Ricard Foye,596,Ricard,31,Sedro-Woolley,Washington,ENTJ,13,24,14,14th voted out,7th jury member,Ua,NaN,NaN,Via Kana,9,5


**Exercise1:** Change every column name of every DataFrame to lowercase and snake case.  This is a standard first step for some programmers as lowercase makes it easier to write and snake case makes it easier to copy multiple-word column names.

For example, `Castaway Id` should end up being `castaway_id`.  You should try doing this using a `for` loop instead of manually changing the names for each column.  It should take you no more than a few lines of code.  Use stackoverflow if you need help.

In [5]:
df_list = [castaway_details,castaways,challenge_description,confessionals,hidden_idols,jury_votes,tribe_mapping,viewers,
          vote_history,season_summary,season_palettes,tribe_colours,challenge_results]

for df in df_list:
    df.rename(columns=lambda col: col.replace(' ', '_').lower(), inplace=True)

**Q2:** What contestant was the oldest at the time of their season?  We want to look at their age at the time of the season and NOT their current age.  Select their row from the `castaway_details` DataFrame and save this as `Q2`.  This should return a DataFrame and the index and missing values should be left as is.

In [6]:
castaways.head()

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
0,Survivor: 41,41,Erika Casupanan,594,Erika,32,Toronta,Ontario,ENFP,13,26,18,Sole Survivor,NaN,Luvu,NaN,NaN,Via Kana,2,8
1,Survivor: 41,41,Deshawn Radden,601,Deshawn,26,Miami,Florida,ENTP,13,26,17,Runner-up,NaN,Luvu,NaN,NaN,Via Kana,7,6
2,Survivor: 41,41,Xander Hastings,597,Xander,20,Chicago,Illinois,INFJ,13,26,16,2nd runner-up,NaN,Yase,NaN,NaN,Via Kana,2,6
3,Survivor: 41,41,Heather Aldret,593,Heather,52,Charleston,South Carolina,ISFJ,13,25,15,15th voted out,8th jury member,Luvu,NaN,NaN,Via Kana,4,6
4,Survivor: 41,41,Ricard Foye,596,Ricard,31,Sedro-Woolley,Washington,ENTJ,13,24,14,14th voted out,7th jury member,Ua,NaN,NaN,Via Kana,9,5


In [7]:
castaway_details.head()

,castaway_id,full_name,short_name,date_of_birth,date_of_death,gender,race,ethnicity,occupation,personality_type
0,1,Sonja Christopher,Sonja,1937-01-28,NaT,Female,NaN,NaN,Musician,ENFP
1,2,B.B. Andersen,B.B.,1936-01-18,2013-10-29,Male,NaN,NaN,Real Estate Developer,ESTJ
2,3,Stacey Stillman,Stacey,1972-08-11,NaT,Female,NaN,NaN,Attorney,ENTJ
3,4,Ramona Gray,Ramona,1971-01-20,NaT,Female,Black,NaN,Biochemist/Chemist,ISTJ
4,5,Dirk Been,Dirk,1976-06-15,NaT,Male,NaN,NaN,Dairy Farmer,ISFP


In [8]:
castaways['age'].max() #What was the highest age

75

In [9]:
castaways[castaways['age']==75] #find oldest contestant

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
646,Survivor: All-Stars,8,Rudy Boesch,14,Rudy,75,Virginia Beach,Virginia,ISTJ,2,6,2,2nd voted out,NaN,Saboga,NaN,NaN,NaN,3,0


In [10]:
Q2 = castaway_details[castaway_details['full_name']=='Rudy Boesch']
Q2

,castaway_id,full_name,short_name,date_of_birth,date_of_death,gender,race,ethnicity,occupation,personality_type
13,14,Rudy Boesch,Rudy,1928-01-20,2019-11-01,Male,NaN,NaN,Retired Navy SEAL,ISTJ


**Q3:** What contestant played in the most number of seasons? Select their row from the `castaway_details` DataFrame and save this as `Q3`.  This should return a DataFrame and the index and missing values should be left as is.

In [11]:
castaways['castaway_id'].value_counts()
#look at groupby Seasons and castaway_id

201    6
274    5
55     5
107    4
333    4
      ..
413    1
415    1
416    1
417    1
1      1
Name: castaway_id, Length: 608, dtype: int64

In [12]:
castaways[castaways['castaway_id']==201]
#4 seasons

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
149,Survivor: Game Changers,34,Oscar Lusth,201,Ozzy,34,Venice,California,ISFP,7,24,9,9th voted out,2nd jury member,Nuku,Tavua,NaN,Maku Maku,6,4
352,Survivor: South Pacific,23,Oscar Lusth,201,Ozzy,29,Venice,California,ISFP,15,38,17,17th voted out,9th jury member,Savaii,NaN,NaN,Te Tuna,17,4
360,Survivor: South Pacific,23,Oscar Lusth,201,Ozzy,29,Venice,California,ISFP,9,22,9,9th voted out,NaN,Savaii,NaN,NaN,Te Tuna,17,4
362,Survivor: South Pacific,23,Oscar Lusth,201,Ozzy,29,Venice,California,ISFP,7,18,7,7th voted out,NaN,Savaii,NaN,NaN,NaN,17,4
491,Survivor: Micronesia,16,Oscar Lusth,201,Ozzy,26,Venice,California,ISFP,10,27,12,10th voted out,2nd jury member,Malakal,Malakal,NaN,Dabu,9,2
539,Survivor: Cook Islands,13,Oscar Lusth,201,Ozzy,24,Venice,California,ISFP,16,39,19,Runner-up,NaN,Aitutaki,Aitutaki,Aitutaki,Aitutonga,1,9


In [13]:
castaways[castaways['castaway_id']==274]
#4 seasons

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
28,Survivor: Winners at War,40,Tyson Apostol,274,Tyson,39,Mesa,Arizona,ESTP,10,25,12,12th voted out,9th jury member,Dakal,NaN,NaN,Koru,12,3
35,Survivor: Winners at War,40,Tyson Apostol,274,Tyson,39,Mesa,Arizona,ESTP,4,11,5,5th voted out,NaN,Dakal,NaN,NaN,NaN,12,3
270,Survivor: Blood vs. Water,27,Tyson Apostol,274,Tyson,34,Provo,Utah,ESTP,15,39,23,Sole Survivor,NaN,Galang,Tadhana,NaN,Kasama,2,8
423,Survivor: Heroes vs. Villains,20,Tyson Apostol,274,Tyson,30,Heber City,Utah,ESTP,6,15,6,6th voted out,NaN,Villains,NaN,NaN,NaN,3,4
456,Survivor: Tocantins,18,Tyson Apostol,274,Tyson,29,Heber City,Utah,ESTP,10,27,9,8th voted out,2nd jury member,Timbira,NaN,NaN,Forza,5,6


In [14]:
castaways[castaways['castaway_id']==55]
#5 seasons

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
34,Survivor: Winners at War,40,Rob Mariano,55,Boston Rob,43,Pensacola,Florida,ESTJ,5,14,6,6th voted out,4th jury member,Sele,Yara,NaN,NaN,4,2
369,Survivor: Redemption Island,22,Rob Mariano,55,Boston Rob,34,Pensacola,Florida,ESTJ,15,39,20,Sole Survivor,NaN,Ometepe,NaN,NaN,Murlonio,7,9
421,Survivor: Heroes vs. Villains,20,Rob Mariano,55,Boston Rob,33,Pensacola,Florida,ESTJ,7,18,8,8th voted out,NaN,Villains,NaN,NaN,NaN,5,5
631,Survivor: All-Stars,8,Rob Mariano,55,Boston Rob,27,Canton,Massachusetts,ESTJ,17,39,17,Runner-up,NaN,Chapera,Chapera,Mogo Mogo,Chaboga Mogo,1,10
707,Survivor: Marquesas,4,Rob Mariano,55,Boston Rob,25,Canton,Massachusetts,ESTJ,7,21,7,7th voted out,NaN,Maraamu,Rotu,NaN,Soliantu,8,2


In [15]:
Q3 = castaway_details[castaway_details['castaway_id']==55]
Q3

,castaway_id,full_name,short_name,date_of_birth,date_of_death,gender,race,ethnicity,occupation,personality_type
54,55,Rob Mariano,Boston Rob,1975-12-25,NaT,Male,NaN,NaN,Construction Worker,ESTJ


**Q4:** Create a DataFrame of all the contestants that won their season (aka their final result in the `castaways` DataFrame was the 'Sole Survivor').  Call this DataFrame `sole_survivor`.  Note that contestants may appear more than one time in this DataFrame if they won more than one season.  Make sure that the index goes from 0 to n-1 and that the DataFrame is sorted ascending by season number.

The DataFrame should have the same columns, and the columns should be in the same order, as the `castaways` DataFrame.

In [16]:
sole_survivor = castaways[castaways['result']=='Sole Survivor'] #find all sole survivors
sole_survivor.sort_values(by=['season'], ascending = True, inplace = True) #put in ascending order
sole_survivor = sole_survivor.reset_index(drop=True) #reset index
sole_survivor

C:\Users\coltm\AppData\Local\Temp\ipykernel_13184\70065992.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sole_survivor.sort_values(by=['season'], ascending = True, inplace = True) #put in ascending order


,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
0,Survivor: Borneo,1,Richard Hatch,16,Richard,39,Newport,Rhode Island,ENTP,14,39,16,Sole Survivor,NaN,Tagi,NaN,NaN,Rattana,6,4
1,Survivor: The Australian Outback,2,Tina Wesson,32,Tina,40,Knoxville,Tennessee,ESFJ,16,42,16,Sole Survivor,NaN,Ogakor,NaN,NaN,Barramundi,0,2
2,Survivor: Africa,3,Ethan Zohn,48,Ethan,27,Lexington,Massachusetts,ISFP,15,39,16,Sole Survivor,NaN,Boran,Boran,NaN,Moto Maji,0,4
3,Survivor: Marquesas,4,Vecepia Towery,64,Vecepia,36,Hayward,California,ISTJ,15,39,16,Sole Survivor,NaN,Maraamu,Rotu,NaN,Soliantu,2,4
4,Survivor: Thailand,5,Brian Heidik,80,Brian,34,Quartz Hill,California,ISTP,15,39,16,Sole Survivor,NaN,Chuay Gahn,NaN,NaN,Chuay Jai,0,8
5,Survivor: The Amazon,6,Jenna Morasca,96,Jenna,21,Bridgeville,Pennsylvania,ISTP,15,39,16,Sole Survivor,NaN,Jaburu,Jaburu,NaN,Jacaré,3,7
6,Survivor: Pearl Islands,7,Sandra Diaz-Twine,112,Sandra,29,Fort Lewis,Washington,ESTP,15,39,18,Sole Survivor,NaN,Drake,Drake,NaN,Balboa,0,3
7,Survivor: All-Stars,8,Amber Brkich,27,Amber,25,Beaver,Pennsylvania,ISFP,17,39,18,Sole Survivor,NaN,Chapera,Chapera,Chapera,Chaboga Mogo,6,6
8,Survivor: Vanuatu,9,Chris Daugherty,130,Chris,33,South Vienna,Ohio,ENTP,15,39,18,Sole Survivor,NaN,Lopevi,Lopevi,NaN,Alinta,3,6
9,Survivor: Palau,10,Tom Westman,150,Tom,40,Sayville,New York,ESTJ,15,39,20,Sole Survivor,NaN,Koror,NaN,NaN,Koror,0,12


**Q5:** Have any contestants won more than one time?  If so, select their records from the `sole_survivor` DataFrame, sorting the rows by season.  Save this as `Q5`.  If no contestant has won twice, save `Q5` as the string `None`.

In [17]:
sole_survivor['castaway_id'].value_counts().head(2)

112    2
424    2
Name: castaway_id, dtype: int64

In [18]:
Q5 = sole_survivor[(sole_survivor['castaway_id']==112) | (sole_survivor['castaway_id']==424)]
Q5

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
6,Survivor: Pearl Islands,7,Sandra Diaz-Twine,112,Sandra,29,Fort Lewis,Washington,ESTP,15,39,18,Sole Survivor,NaN,Drake,Drake,NaN,Balboa,0,3
19,Survivor: Heroes vs. Villains,20,Sandra Diaz-Twine,112,Sandra,35,Fayetteville,North Carolina,ESTP,15,39,20,Sole Survivor,NaN,Villains,NaN,NaN,Yin Yang,3,4
27,Survivor: Cagayan,28,Tony Vlachos,424,Tony,39,Jersey City,New Jersey,ESTP,14,39,18,Sole Survivor,NaN,Aparri,Solana,NaN,Solarrion,5,6
39,Survivor: Winners at War,40,Tony Vlachos,424,Tony,45,Allendale,New Jersey,ESTP,15,39,22,Sole Survivor,NaN,Dakal,Dakal,NaN,Koru,0,9


**Q6:** What is the average age of contestants when they appeared on the show?  Save this as `Q6`.  Round to nearest integer.

In [19]:
Q6 = round(castaways['age'].mean())
Q6

33

**Q7:** Who played the most total number of days of Survivor? If a contestant appeared on more than one season, you would add their total days for each season together. Save the top five contestants in terms of total days played as a DataFrame and call it `Q7`, sorted in descending order by total days played.  

The following columns should be included: `castaway_id`, `full_name`, and `total_days_played` where `total_days_played` is the sum of all days a contestant played. The index should go from 0 to n-1.

Note:  Be careful because on some seasons, the contestant was allowed to come back into the game after being voted off.  Take a look at [Season 23's contestant Oscar Lusth](https://en.wikipedia.org/wiki/Ozzy_Lusth#South_Pacific) in the `castaways` DataFrame as an example.  He was voted out 7th and then returned to the game.  He was then voted out 9th and returned to the game a second time.  He was then voted out 17th the final time.  Be aware of this in your calculations and make sure you are counting the days according to the last time they were voted off or won. 

In [20]:
castaways[castaways['full_name']=='Oscar Lusth']

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
149,Survivor: Game Changers,34,Oscar Lusth,201,Ozzy,34,Venice,California,ISFP,7,24,9,9th voted out,2nd jury member,Nuku,Tavua,NaN,Maku Maku,6,4
352,Survivor: South Pacific,23,Oscar Lusth,201,Ozzy,29,Venice,California,ISFP,15,38,17,17th voted out,9th jury member,Savaii,NaN,NaN,Te Tuna,17,4
360,Survivor: South Pacific,23,Oscar Lusth,201,Ozzy,29,Venice,California,ISFP,9,22,9,9th voted out,NaN,Savaii,NaN,NaN,Te Tuna,17,4
362,Survivor: South Pacific,23,Oscar Lusth,201,Ozzy,29,Venice,California,ISFP,7,18,7,7th voted out,NaN,Savaii,NaN,NaN,NaN,17,4
491,Survivor: Micronesia,16,Oscar Lusth,201,Ozzy,26,Venice,California,ISFP,10,27,12,10th voted out,2nd jury member,Malakal,Malakal,NaN,Dabu,9,2
539,Survivor: Cook Islands,13,Oscar Lusth,201,Ozzy,24,Venice,California,ISFP,16,39,19,Runner-up,NaN,Aitutaki,Aitutaki,Aitutaki,Aitutonga,1,9


**Q8A & Q8B**: Using the `castaway_details` data, what is the percentage of total extroverts and introverts that have played the game (count players only once even if they have played in more than one season).  Do not count contestants without a personality type listed in your calculations.  Save these percentages as `Q8A` and `Q8B` respectively.  Note: Round all percentages to two decimal points and write as a float (example: 55.57).  

For more information on personality types check this [Wikipedia article](https://en.wikipedia.org/wiki/Myers%E2%80%93Briggs_Type_Indicator).

In [21]:
castaways_details2 = castaway_details.copy()#.drop_duplicates(subset=['castaway_id'])
castaways_details2
#Extroverts = ['ESTP','ESFP','ENFP','ENTP','ESTJ', 'ESFJ','ENFJ','ENTJ']
#Introverts = ['ISTJ','ISFJ','INFJ','INTJ','ISTP','ISFP','INFP','INTP']
#castaways_details2[castaways_details2['full_name']=='Oscar Lusth']
castaways_details2 = castaways_details2.dropna(subset = ['personality_type']) 
#drop na values in the personality_type column

,castaway_id,full_name,short_name,date_of_birth,date_of_death,gender,race,ethnicity,occupation,personality_type
0,1,Sonja Christopher,Sonja,1937-01-28,NaT,Female,NaN,NaN,Musician,ENFP
1,2,B.B. Andersen,B.B.,1936-01-18,2013-10-29,Male,NaN,NaN,Real Estate Developer,ESTJ
2,3,Stacey Stillman,Stacey,1972-08-11,NaT,Female,NaN,NaN,Attorney,ENTJ
3,4,Ramona Gray,Ramona,1971-01-20,NaT,Female,Black,NaN,Biochemist/Chemist,ISTJ
4,5,Dirk Been,Dirk,1976-06-15,NaT,Male,NaN,NaN,Dairy Farmer,ISFP
...,...,...,...,...,...,...,...,...,...,...
603,604,Tiffany Seely,Tiffany,1973-12-08,NaT,Female,White,Jewish,Teacher,ENTP
604,605,Sydney Segal,Sydney,1995-07-19,NaT,Female,White,Jewish,Law Student,ESTP
605,606,Shantel Smith,Shan,1987-03-11,NaT,Female,Black,NaN,Pastor,ENFJ
606,607,David Voce,Voce,1986-05-01,NaT,Male,NaN,NaN,Neurosurgeon,ENTJ


In [62]:

castaways_details2 = castaways_details2.dropna(subset = ['personality_type'])

In [63]:
#castaway_personality_counts = castaways_details2['personality_type'].value_counts()
#castaway_personality_counts
cd_extro = castaways_details2[(castaways_details2['personality_type']=='ESTP')|(castaways_details2['personality_type']=='ESFP')|
                             (castaways_details2['personality_type']=='ENFP')|(castaways_details2['personality_type']=='ENTP')|
                             (castaways_details2['personality_type']=='ESTJ')|(castaways_details2['personality_type']=='ESFJ')|
                             (castaways_details2['personality_type']=='ENFJ')|(castaways_details2['personality_type']=='ENTJ')]
Q8A = round((len(cd_extro)/len(castaways_details2))*100,2)
Q8A

53.63

In [64]:
cd_intro = castaways_details2[(castaways_details2['personality_type']=='ISTJ')|(castaways_details2['personality_type']=='ISFJ')|
                             (castaways_details2['personality_type']=='INFJ')|(castaways_details2['personality_type']=='INTJ')|
                             (castaways_details2['personality_type']=='ISTP')|(castaways_details2['personality_type']=='ISFP')|
                             (castaways_details2['personality_type']=='INFP')|(castaways_details2['personality_type']=='INTP')]
Q8B = round((len(cd_intro)/len(castaways_details2))*100,2)
Q8B

46.37

**Q9A & Q9B**: Now that we know the percentages of total players that are extroverted and introverted, let's see if that made a difference in terms of who actually won their season.

What is the percentage of total extroverts and introverts that have won the game (count players only once even if they have won more than one season)?  Save these percentages as `Q9A` and `Q9B` respectively.  Note: Round all percentages to two decimal points and write as a float (example: 55.57).

In [25]:
sole_survivor_pt = sole_survivor.copy().drop_duplicates(subset=['castaway_id','personality_type'])
sole_survivor_pt #season 20 and 40 winners dropped as duplicates

,season_name,season,full_name,castaway_id,castaway,age,city,state,personality_type,episode,day,order,result,jury_status,original_tribe,swapped_tribe,swapped_tribe_2,merged_tribe,total_votes_received,immunity_idols_won
0,Survivor: Borneo,1,Richard Hatch,16,Richard,39,Newport,Rhode Island,ENTP,14,39,16,Sole Survivor,NaN,Tagi,NaN,NaN,Rattana,6,4
1,Survivor: The Australian Outback,2,Tina Wesson,32,Tina,40,Knoxville,Tennessee,ESFJ,16,42,16,Sole Survivor,NaN,Ogakor,NaN,NaN,Barramundi,0,2
2,Survivor: Africa,3,Ethan Zohn,48,Ethan,27,Lexington,Massachusetts,ISFP,15,39,16,Sole Survivor,NaN,Boran,Boran,NaN,Moto Maji,0,4
3,Survivor: Marquesas,4,Vecepia Towery,64,Vecepia,36,Hayward,California,ISTJ,15,39,16,Sole Survivor,NaN,Maraamu,Rotu,NaN,Soliantu,2,4
4,Survivor: Thailand,5,Brian Heidik,80,Brian,34,Quartz Hill,California,ISTP,15,39,16,Sole Survivor,NaN,Chuay Gahn,NaN,NaN,Chuay Jai,0,8
5,Survivor: The Amazon,6,Jenna Morasca,96,Jenna,21,Bridgeville,Pennsylvania,ISTP,15,39,16,Sole Survivor,NaN,Jaburu,Jaburu,NaN,Jacaré,3,7
6,Survivor: Pearl Islands,7,Sandra Diaz-Twine,112,Sandra,29,Fort Lewis,Washington,ESTP,15,39,18,Sole Survivor,NaN,Drake,Drake,NaN,Balboa,0,3
7,Survivor: All-Stars,8,Amber Brkich,27,Amber,25,Beaver,Pennsylvania,ISFP,17,39,18,Sole Survivor,NaN,Chapera,Chapera,Chapera,Chaboga Mogo,6,6
8,Survivor: Vanuatu,9,Chris Daugherty,130,Chris,33,South Vienna,Ohio,ENTP,15,39,18,Sole Survivor,NaN,Lopevi,Lopevi,NaN,Alinta,3,6
9,Survivor: Palau,10,Tom Westman,150,Tom,40,Sayville,New York,ESTJ,15,39,20,Sole Survivor,NaN,Koror,NaN,NaN,Koror,0,12


In [26]:
ss_intro = sole_survivor_pt[(sole_survivor_pt['personality_type']=='ISTJ')|(sole_survivor_pt['personality_type']=='ISFJ')|
                             (sole_survivor_pt['personality_type']=='INFJ')|(sole_survivor_pt['personality_type']=='INTJ')|
                             (sole_survivor_pt['personality_type']=='ISTP')|(sole_survivor_pt['personality_type']=='ISFP')|
                             (sole_survivor_pt['personality_type']=='INFP')|(sole_survivor_pt['personality_type']=='INTP')]
Q9B = round((len(ss_intro)/len(sole_survivor_pt))*100,2)
Q9B

38.46

In [27]:
ss_extro = sole_survivor_pt[(sole_survivor_pt['personality_type']=='ESTP')|(sole_survivor_pt['personality_type']=='ESFP')|
                             (sole_survivor_pt['personality_type']=='ENFP')|(sole_survivor_pt['personality_type']=='ENTP')|
                             (sole_survivor_pt['personality_type']=='ESTJ')|(sole_survivor_pt['personality_type']=='ESFJ')|
                             (sole_survivor_pt['personality_type']=='ENFJ')|(sole_survivor_pt['personality_type']=='ENTJ')]
Q9A = round((len(ss_extro)/len(sole_survivor_pt))*100,2)
Q9A

61.54

**Q10:** Which contestants have never received a tribal council vote (i.e. a vote to be voted out of the game as shown in the `vote_id` column in the `vote_history` DataFrame)? Note that there are various reasons for a contestant to not receive a tribal vote: they quit, made it to the end, medical emergency, etc.  Select their rows from the `castaway_details` DataFrame and save this as `Q10` in ascending order by `castaway_id`.  This should return a DataFrame and the index and missing values should be left as is.

In [28]:
no_votes = vote_history[vote_history.vote_id.isnull()]
no_votes

,season_name,season,episode,day,tribe_status,castaway,immunity,vote,nullified,voted_out,order,vote_order,castaway_id,vote_id,voted_out_id
16,Survivor: 41,41,2,5,Original,Xander,NaN,None,False,Voce,3,1,597,NaN,607.0
17,Survivor: 41,41,3,7,Original,Brad,NaN,None,False,Brad,4,1,602,NaN,602.0
41,Survivor: 41,41,7,14,Merged,Sydney,NaN,Shot in the dark,False,Sydney,7,1,605,NaN,605.0
60,Survivor: 41,41,9,17,Merged,Heather,NaN,None,False,Naseer,9,2,593,NaN,600.0
61,Survivor: 41,41,9,17,Merged,Naseer,NaN,None,False,Naseer,9,2,600,NaN,600.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4652,Survivor: The Australian Outback,2,15,41,Merged,Tina,NaN,None,False,Keith,14,1,32,NaN,30.0
4745,Survivor: Borneo,1,13,37,Merged,Richard,NaN,None,False,Sue,13,2,16,NaN,13.0
4747,Survivor: Borneo,1,13,37,Merged,Sue,NaN,None,False,Sue,13,2,13,NaN,13.0
4749,Survivor: Borneo,1,13,38,Merged,Richard,NaN,None,False,Rudy,14,1,16,NaN,14.0


In [29]:
Q10 = pd.merge(castaway_details, no_votes, left_on='castaway_id',right_on='castaway_id', how='right')
Q10 = Q10[['castaway_id', 'full_name','short_name', 'date_of_birth', 'date_of_death', 'gender','race','ethnicity','occupation',
         'personality_type']]
Q10

,castaway_id,full_name,short_name,date_of_birth,date_of_death,gender,race,ethnicity,occupation,personality_type
0,597,Xander Hastings,Xander,2000-08-11,NaT,Male,NaN,NaN,App Developer,INFJ
1,602,Brad Reese,Brad,1971-07-21,NaT,Male,NaN,NaN,Rancher,ESFP
2,605,Sydney Segal,Sydney,1995-07-19,NaT,Female,White,Jewish,Law Student,ESTP
3,593,Heather Aldret,Heather,1969-03-01,NaT,Female,NaN,NaN,Stay-at-home Mom,ISFJ
4,600,Naseer Muttalif,Naseer,1984-06-21,NaT,Male,Asian,Sri Lankan,Sales Manager,ESFJ
...,...,...,...,...,...,...,...,...,...,...
218,32,Tina Wesson,Tina,1960-12-26,NaT,Female,NaN,NaN,Personal Nurse;Motivational Speaker,ESFJ
219,16,Richard Hatch,Richard,1961-04-08,NaT,Male,NaN,NaN,Corporate Trainer,ENTP
220,13,Susan Hawk,Sue,1961-08-17,NaT,Female,NaN,NaN,Truck Driver,ESTP
221,16,Richard Hatch,Richard,1961-04-08,NaT,Male,NaN,NaN,Corporate Trainer,ENTP


**Q11:** What contestant has won the most number of challenges?  Select their row from the `castaway_details` DataFrame and save this as `Q11`.  This should return a DataFrame and the index and missing values should be left as is.

In [30]:
challenge_results['winner_id'].value_counts()

201.0    42
55.0     37
197.0    34
451.0    33
333.0    31
         ..
445.0     1
324.0     1
537.0     1
304.0     1
305.0     1
Name: winner_id, Length: 564, dtype: int64

In [31]:
Q11 = castaway_details[castaway_details['castaway_id']==201]
Q11

,castaway_id,full_name,short_name,date_of_birth,date_of_death,gender,race,ethnicity,occupation,personality_type
200,201,Oscar Lusth,Ozzy,1981-08-23,NaT,Male,Mexican American,Hispanic or Latino,Waiter;Photographer,ISFP


**Q12:** Let's see how many winners ended up getting unanimous jury votes to win the game.  Create a Dataframe that shows the survivors that got unanimous jury votes with these columns in the final output: `season`, `season_name`, `winner_id`, `full_name`.  The DataFrame should be sorted by season and the index should go from 0 to n-1.  Save this as `Q12`. 

In [32]:
season_summary

,season_name,season,location,country,tribe_setup,full_name,winner_id,winner,runner_ups,final_vote,timeslot,premiered,ended,filming_started,filming_ended,viewers_premier,viewers_finale,viewers_reunion,viewers_mean,rank
0,Survivor: Borneo,1,"Pulau Tiga, Sabah, Malaysia",Malaysia,Two tribes of eight new players,Richard Hatch,16,Richard,Kelly Wiglesworth,4-3,Wednesday 8:00 pm,2000-05-31,2000-08-23,2000-03-13,2000-04-20,15.51,51.69,36.70,28.30,2.0
1,Survivor: The Australian Outback,2,"Herbert River at Goshen Station, Queensland, A...",Australia,Two tribes of eight new players,Tina Wesson,32,Tina,Colby Donaldson,4-3,Thursday 8:00 pm,2001-01-28,2001-05-03,2000-10-23,2000-12-03,45.37,36.35,28.01,29.80,1.0
2,Survivor: Africa,3,"Shaba National Reserve, Kenya",Kenya,Two tribes of eight new players,Ethan Zohn,48,Ethan,Kim Johnson,5-2,Thursday 8:00 pm,2001-10-11,2002-01-10,2001-07-11,2001-08-18,23.84,27.26,19.05,20.69,8.0
3,Survivor: Marquesas,4,"Nuku Hiva, Marquesas Islands, French Polynesia",Polynesia,Two tribes of eight new players,Vecepia Towery,64,Vecepia,Neleh Dennis,4-3,Thursday 8:00 pm,2002-02-28,2002-05-19,2001-11-12,2001-12-20,23.19,25.87,19.05,20.77,6.0
4,Survivor: Thailand,5,"Ko Tarutao, Satun Province, Thailand",Thailand,Two tribes of eight new players; picked by the...,Brian Heidik,80,Brian,Clay Jordan,4-3,Thursday 8:00 pm,2002-09-19,2002-12-19,2002-06-10,2002-07-18,23.05,24.08,20.43,21.21,4.0
5,Survivor: The Amazon,6,"Rio Negro, Amazonas, Brazil",Brazil,Two tribes of eight new players divided by gender,Jenna Morasca,96,Jenna,Matthew Von Ertfelda,6-1,Thursday 8:00 pm,2003-02-13,2003-05-11,2002-11-07,2002-12-15,23.26,22.29,20.43,19.97,9.0
6,Survivor: Pearl Islands,7,"Pearl Islands, Panama",Panama,Two tribes of eight new players,Sandra Diaz-Twine,112,Sandra,Lillian Morris,6-1,Thursday 8:00 pm,2003-09-18,2003-12-14,2003-06-23,2003-07-31,21.50,25.23,21.87,20.72,7.0
7,Survivor: All-Stars,8,"Pearl Islands, Panama",Panama,Three tribes of six returning players,Amber Brkich,27,Amber,Rob Mariano,4-3,Thursday 8:00 pm,2004-02-01,2004-05-09,2003-11-03,2003-12-11,33.53,24.76,21.87,21.49,3.0
8,Survivor: Vanuatu,9,"Efate, Shefa, Vanuatu",Vanuatu,Two tribes of nine new players divided by gender,Chris Daugherty,130,Chris,Twila Tanner,5-2,Thursday 8:00 pm,2004-09-16,2004-12-12,2004-06-28,2004-08-05,20.06,19.72,15.23,19.64,10.0
9,Survivor: Palau,10,"Koror, Palau",Palau,A schoolyard pick of two tribes of nine new pl...,Tom Westman,150,Tom,Katie Gallagher,6-1,Thursday 8:00 pm,2005-02-17,2005-05-15,2004-10-27,2004-12-04,23.66,20.80,15.23,20.91,5.0


In [33]:
unanimous = season_summary[(season_summary['winner_id']==221)|(season_summary['winner_id']==281)|
                             (season_summary['winner_id']==348)|(season_summary['winner_id']==433)|
                             (season_summary['winner_id']==498)]
unanimous

,season_name,season,location,country,tribe_setup,full_name,winner_id,winner,runner_ups,final_vote,timeslot,premiered,ended,filming_started,filming_ended,viewers_premier,viewers_finale,viewers_reunion,viewers_mean,rank
13,Survivor: Fiji,14,"Macuata, Vanua Levu, Fiji",Fiji,Two tribes of nine new players divided by one ...,Earl Cole,221,Earl,"Cassandra Franklin, Andria Herd",9-0-0,Thursday 8:00 pm,2007-02-08,2007-05-13,2006-10-30,2006-12-07,16.68,13.63,13.53,14.83,15.0
17,Survivor: Tocantins,18,"Jalapao, Tocantins, Brazil",Brazil,Two tribes of eight new players,James Thomas Jr.,281,J.T.,Stephen Fishbach,7-0,Thursday 8:00 pm,2009-02-12,2009-05-17,2008-11-02,2008-12-10,13.63,12.94,11.74,12.86,19.0
25,Survivor: Caramoan,26,"Caramoan, Camarines Sur, Philippines",Philippines,Two tribes of ten: new players against past co...,John Cochran,348,Cochran,"Dawn Meehan, Sherri Biethman",8-0-0,Wednesday 8:00 pm,2013-02-13,2013-05-12,2012-05-21,2012-06-28,8.94,10.16,8.77,10.82,28.0
30,Survivor: Cambodia,31,"Koh Rong, Cambodia",Cambodia,Two tribes of ten returning players who only p...,Jeremy Collins,433,Jeremy,"Spencer Bledsoe, Tasha Fox",10-0-0,Wednesday 8:00 pm,2015-09-23,2015-12-16,2015-05-31,2015-07-08,9.70,9.45,6.49,10.99,26.0
32,Survivor: Millennials vs. Gen X,33,"Mamanuca Islands, Fiji",Fiji,Two tribes of ten new players divided by gener...,Adam Klein,498,Adam,"Hannah Shapiro, Ken McNickle",10-0-0,Wednesday 8:00 pm,2016-09-21,2016-12-14,2016-04-04,2016-05-12,9.46,9.09,6.40,10.32,24.0


In [34]:
Q12 = unanimous[['season', 'season_name', 'winner_id', 'full_name']]
Q12 = Q12.reset_index(drop=True)
Q12

,season,season_name,winner_id,full_name
0,14,Survivor: Fiji,221,Earl Cole
1,18,Survivor: Tocantins,281,James Thomas Jr.
2,26,Survivor: Caramoan,348,John Cochran
3,31,Survivor: Cambodia,433,Jeremy Collins
4,33,Survivor: Millennials vs. Gen X,498,Adam Klein


**Q13:** For the `castaway_details` DataFrame, there is a `full_name` column and a `short_name` column.  It would be helpful for future analysis to have the contestants first and last name split into separate columns.  First copy the `castaway_details` DataFrame to a new DataFrame called `Q13` so that we do not change the original DataFrame.  

Create two new columns and add the contestant's first name to a new column called `first_name` and their last name to a new column called `last_name`.  Add these columns to the `Q13` DataFrame.  Put the `first_name` and `last_name` columns between the `full_name` and `short_name` columns.

Note:  Be careful as some players have last names with multiple spaces.  For example, `Lex van den Berghe`.  You should code `Lex` as his first name and `van den Berghe` as his last name.

In [35]:
Q13 = castaway_details.copy()

In [36]:
#Q13[['first_name','last_name']]=Q13['full_name'].str.split(' ')

In [37]:
Q13['first_name'] = Q13['full_name'].apply(lambda x: x.split()[0])

In [38]:
Q13['last_name'] = Q13['full_name'].apply(lambda x: x.split()[1])

In [39]:
Q13 = Q13[['castaway_id','full_name','first_name','last_name','short_name','date_of_birth','date_of_death','gender',
           'race','ethnicity','occupation','personality_type']]
Q13

,castaway_id,full_name,first_name,last_name,short_name,date_of_birth,date_of_death,gender,race,ethnicity,occupation,personality_type
0,1,Sonja Christopher,Sonja,Christopher,Sonja,1937-01-28,NaT,Female,NaN,NaN,Musician,ENFP
1,2,B.B. Andersen,B.B.,Andersen,B.B.,1936-01-18,2013-10-29,Male,NaN,NaN,Real Estate Developer,ESTJ
2,3,Stacey Stillman,Stacey,Stillman,Stacey,1972-08-11,NaT,Female,NaN,NaN,Attorney,ENTJ
3,4,Ramona Gray,Ramona,Gray,Ramona,1971-01-20,NaT,Female,Black,NaN,Biochemist/Chemist,ISTJ
4,5,Dirk Been,Dirk,Been,Dirk,1976-06-15,NaT,Male,NaN,NaN,Dairy Farmer,ISFP
...,...,...,...,...,...,...,...,...,...,...,...,...
603,604,Tiffany Seely,Tiffany,Seely,Tiffany,1973-12-08,NaT,Female,White,Jewish,Teacher,ENTP
604,605,Sydney Segal,Sydney,Segal,Sydney,1995-07-19,NaT,Female,White,Jewish,Law Student,ESTP
605,606,Shantel Smith,Shantel,Smith,Shan,1987-03-11,NaT,Female,Black,NaN,Pastor,ENFJ
606,607,David Voce,David,Voce,Voce,1986-05-01,NaT,Male,NaN,NaN,Neurosurgeon,ENTJ


**Q14:** Let's say that we wanted to predict a contestants personality type based on the information in the data files.  Your task is to create a DataFrame that lists the `castaway_id`, `full_name` and `personality_type` for each castaway contestant.  However, since most machine learning algorithms use numeric data, you want to change the personality types to the following numbers:
- ISTJ - 1
- ISTP - 2
- ISFJ - 3
- ISFP - 4
- INFJ - 5
- INFP - 6
- INTJ - 7
- INTP - 8
- ESTP - 9
- ESTJ - 10
- ESFP - 11
- ESFJ - 12
- ENFP - 13
- ENFJ - 14
- ENTP - 15
- ENTJ - 16
- Missing values - 17

Save this new DataFrame as `Q14` and sort based on `castaway_id` in ascending order.

In [40]:
Q14=castaway_details.copy()

In [41]:
Q14 = Q14[['castaway_id','full_name','personality_type']]

In [42]:
Q14.replace(['ISTJ'],'1',inplace=True)
Q14.replace(['ISTP'],'2',inplace=True)
Q14.replace(['ISFJ'],'3',inplace=True)
Q14.replace(['ISFP'],'4',inplace=True)
Q14.replace(['INFJ'],'5',inplace=True)
Q14.replace(['INFP'],'6',inplace=True)
Q14.replace(['INTJ'],'7',inplace=True)
Q14.replace(['INTP'],'8',inplace=True)
Q14.replace(['ESTP'],'9',inplace=True)
Q14.replace(['ESTJ'],'10',inplace=True)
Q14.replace(['ESFP'],'11',inplace=True)
Q14.replace(['ESFJ'],'12',inplace=True)
Q14.replace(['ENFP'],'13',inplace=True)
Q14.replace(['ENFJ'],'14',inplace=True)
Q14.replace(['ENTP'],'15',inplace=True)
Q14.replace(['ENTJ'],'16',inplace=True)
Q14=Q14.fillna('17')

In [43]:
Q14

,castaway_id,full_name,personality_type
0,1,Sonja Christopher,13
1,2,B.B. Andersen,10
2,3,Stacey Stillman,16
3,4,Ramona Gray,1
4,5,Dirk Been,4
...,...,...,...
603,604,Tiffany Seely,15
604,605,Sydney Segal,9
605,606,Shantel Smith,14
606,607,David Voce,16


**Q15:** After thinking about this some more, you realize that you don't want to code the personality traits as you did in problem 14 since the data is not ordinal.  Some machine learning algorithms will assume that numbers close to each other are more alike than those that are away from each other and that is not the case with these personality types.

Let's create a new DataFrame called `Q15` that creates dummy columns (using `get_dummies`) for the original personality type column.  Add a prefix called "type" and drop the first column to help prevent multicollinearity.  The columns should be `castaway_id`, `full_name` followed by the various dummy columns for the personality types.  Don't worry about any missing values in this step.

Remember: Don't change any of the original DataFrames or CodeGrade will not work correctly for this assignment.  Make sure you use `copy()` if needed.

In [44]:
Q15=castaway_details.copy()

In [45]:
Q15=Q15[['castaway_id','full_name','personality_type']]

In [46]:
Q15 = pd.get_dummies(Q15,prefix='type',columns=['personality_type'],drop_first=True)
Q15

,castaway_id,full_name,type_ENFP,type_ENTJ,type_ENTP,type_ESFJ,type_ESFP,type_ESTJ,type_ESTP,type_INFJ,type_INFP,type_INTJ,type_INTP,type_ISFJ,type_ISFP,type_ISTJ,type_ISTP
0,1,Sonja Christopher,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,B.B. Andersen,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,3,Stacey Stillman,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0
3,4,Ramona Gray,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
4,5,Dirk Been,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
603,604,Tiffany Seely,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
604,605,Sydney Segal,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0
605,606,Shantel Smith,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
606,607,David Voce,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


**Q16:** After running your data above through your machine learning model, you determine that a better prediction might come from breaking the personality type into its four parts (one part for each character in the type).  Your task is now to create a DataFrame called `Q16` that splits the personality type into the various parts and creates a new column for each part (these columns should be called `interaction` that will represent the first letter in the personality type, `information` for the second letter, `decision` for the third, and `organization` for the fourth).

Again, since most machine learning algorithms work with numeric data, perform the following on the four new columns:
- `interaction` --> code all `I`'s as `0` and `E`'s as `1`
- `information` --> code all `S`'s as `0` and `N`'s as `1`
- `decision` --> code all `T`'s as `0` and `F`'s as `1`
- `organization` --> code as `J`'s with `0` and `P`'s as `1`
- Any missing values should be coded with a `2`
- Double check that all of the above columns are an integer type.  Some students have a problem with CodeGrade because one of their columns ends up being a string instead of an int.

For example, if a contestant's personality type was `ENTJ`, your columns for that row would be:
- `1` for `interaction` because of the `E`
- `1` for `information` because of the `N`
- `0` for `decision` because of the `T` 
- `0` for `organization` because of the `J`

The new DataFrame should be sorted in `castaway_id` order and have the following columns in this order: `castaway_id`, `full_name`, `personality_type`, `interaction`, `information`, `decision`, `organization`.

Remember: Don't change any of the original DataFrames or CodeGrade will not work correctly for this assignment.  Make sure you use `copy()` if needed.

In [93]:
Q16 = castaway_details.copy()
Q16 = Q16[['castaway_id','full_name','personality_type']]
Q16['interaction'] = Q16['personality_type'].astype(str).str[0]
Q16['information'] = Q16['personality_type'].astype(str).str[1]
Q16['decision'] = Q16['personality_type'].astype(str).str[2]
Q16['organization'] = Q16['personality_type'].astype(str).str[3]
Q16 = Q16.replace(np.nan, 2)

In [94]:
Q16['organization'].isna().value_counts()

False    608
Name: organization, dtype: int64

In [95]:
#Q16 = pd.get_dummies(Q16a,columns=['interaction', 'information','decision','organization'],drop_first=True)
#Q16
Q16.replace({'I':0, 'E':1}, inplace=True )
Q16.replace({'S':0, 'N':1}, inplace=True )
Q16.replace({'T':0, 'F':1}, inplace=True )
Q16.replace({'J':0, 'P':1}, inplace=True )
Q16 = Q16.replace(np.nan, 2)
Q16

,castaway_id,full_name,personality_type,interaction,information,decision,organization
0,1,Sonja Christopher,ENFP,1,1,1,1
1,2,B.B. Andersen,ESTJ,1,0,0,0
2,3,Stacey Stillman,ENTJ,1,1,0,0
3,4,Ramona Gray,ISTJ,0,0,0,0
4,5,Dirk Been,ISFP,0,0,1,1
...,...,...,...,...,...,...,...
603,604,Tiffany Seely,ENTP,1,1,0,1
604,605,Sydney Segal,ESTP,1,0,0,1
605,606,Shantel Smith,ENFJ,1,1,1,0
606,607,David Voce,ENTJ,1,1,0,0


**Q17:** Using data from `castaways`, create a DataFrame called `Q17` that bins the contestant ages (their age when they were on the season, not their current age) into the following age categories:
- 18-24
- 25-34
- 35-44
- 45-54
- 55-64
- 65+

The final DataFrame should have the following columns in this order: `season`, `castaway_id`, `full_name`, `age`, and `age_category`.  The DataFrame should be sorted by age and then castaway_id.  The index should be 0 through n-1.  You should have the same amount of rows as in the `castaways` DataFrame.

Remember: Don't change any of the original DataFrames or CodeGrade will not work correctly for this assignment.  Make sure you use `copy()` if needed.

In [49]:
castaway= castaways.copy()
castaway_bins = [17,24,34,44,54,64,100] #create bins
castaway_labels = ['18-24','25-34','35-44','45-54','55-64','65+'] #create bin headers
castaway['age_category'] = pd.cut(x=castaways['age'],bins=castaway_bins,labels=castaway_labels) #putting values into bins
castaway['age_category'] = castaway['age_category'].astype('category')
#castaway['age_category'] = castaway['age_category'].cat.add_categories('unknown') #create unknown column if someone doesnt have age
castaway['age_category'] = castaway['age_category'].cat.reorder_categories(['18-24','25-34','35-44','45-54','55-64','65+'])

In [50]:
Q17 = castaway[['season', 'castaway_id', 'full_name', 'age', 'age_category']]
Q17.sort_values(by=['age','castaway_id'], ascending = True, inplace = True)
Q17 = Q17.reset_index(drop=True)
Q17

C:\Users\coltm\AppData\Local\Temp\ipykernel_13184\64786843.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  Q17.sort_values(by=['age','castaway_id'], ascending = True, inplace = True)


,season,castaway_id,full_name,age,age_category
0,33,491,Will Wahl,18,18-24
1,36,528,Michael Yerger,18,18-24
2,18,270,Spencer Duhm,19,18-24
3,22,336,Natalie Tenerelli,19,18-24
4,23,350,Brandon Hantz,19,18-24
...,...,...,...,...,...
757,24,366,Greg Smith,64,55-64
758,21,304,Jimmy Johnson,67,65+
759,32,474,Joseph Del Campo,71,65+
760,1,14,Rudy Boesch,72,65+


**Q18:** Based on the age categories you created above, what are the normalized percentages for the various age categories using `value_counts()`.  Sort the value counts by index.  Save this as `Q18`.

In [51]:
castaway['age_category']
Q18 = castaway['age_category'].value_counts(normalize=True, sort = False)
Q18

18-24    0.190289
25-34    0.437008
35-44    0.211286
45-54    0.124672
55-64    0.031496
65+      0.005249
Name: age_category, dtype: float64

**Q19:** Which contestant(s) played a perfect game?  A perfect game is considered when the contestant:
- didn't receive any tribal council votes all season (this is different than Q10 since some players played multiple times.  They got voted out in one season so they would not show in Q10 but they came back for another season and didn't receive any tribal council votes)
- won the game
- got unanimous jury votes (see question 12)

Save this DataFrame as `Q19` with the following columns: `season_name`, `season`, `castaway_id`, `full_name`, `tribal_council_votes`, `jury_votes`.  The DataFrame should be sorted by season and the index should be 0 to n-1.

In [52]:
unanimous
final_counts = jury_votes.groupby(['season', 'finalist_id']).sum()
#final_counts.unstack().head(30)
final_counts

vote  castaway_id
season finalist_id                   
1      15              3           77
       16              4           77
2      31              3          189
       32              4          189
3      47              2          301
...                  ...          ...
40     442             4         5032
       478             0         5032
41     594             7         4804
       597             0         4804
       601             1         4804

[108 rows x 2 columns]

In [53]:
no_votes2=castaways[(castaways['total_votes_received']==0)]

In [54]:
Q19 = pd.merge(unanimous,no_votes2)

In [55]:
Q19 = Q19[(Q19['total_votes_received']==0)]
Q19['jury_votes'] = [7,8]
Q19 = Q19.rename(columns={'total_votes_received':'tribal_council_votes'})

In [56]:
Q19 = Q19[['season_name', 'season', 'castaway_id', 'full_name', 'tribal_council_votes', 'jury_votes']]
Q19

,season_name,season,castaway_id,full_name,tribal_council_votes,jury_votes
0,Survivor: Tocantins,18,281,James Thomas Jr.,0,7
1,Survivor: Caramoan,26,348,John Cochran,0,8
